Pehle ek design decision. Plan kehta hai ki target wahi Churned column ho jo Day 35B mein banaya tha. Hum label ki definition same rakhenge (90 din tak koi order nahi), lekin use future window par apply karenge. Isse leakage hat jaata hai aur Recency bhi ek legit feature ban jaata hai.

Pehle problem samjho: Abhi Churned aur Recency dono ek hi time window (poora dataset) se bane hain. Model ko “churned hai ya nahi” batane ke liye bas wahi rule dobara seekhna padta hai. Sirf Recency drop karna kaafi nahi hai, kyunki Frequency aur Monetary bhi us window ke hain jisme churn decide hua. Aur Recency hatane se sabse useful signal bhi chala jaata hai.

- Fix: observation window aur prediction window alag karo

    - Cutoff date = dataset ki last date minus 90 din
    - Features: sirf cutoff se pehle ke transactions se bante hain (past behaviour)
    - Label: Churned = 1 agar customer ne cutoff ke baad ke 90 din mein koi order nahi kiya

Ab model past dekh kar future predict karta hai, jo asli business problem hai.

In [13]:
import sklearn
print(sklearn.__version__)

1.9.1


2. Data load karo aur cutoff nikaalo:

In [14]:
import pandas as pd
import numpy as np

tx = pd.read_csv(
    '../../data/processed/step4_datatypes_fixed.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
tx['LineTotal'] = tx['Quantity'] * tx['Price']

end_date = tx['InvoiceDate'].max()
cutoff = end_date - pd.Timedelta(days=90)
print(f"Dataset end: {end_date}, Cutoff: {cutoff}")

past = tx[tx['InvoiceDate'] < cutoff]
future = tx[tx['InvoiceDate'] >= cutoff]
print(f"Past rows: {len(past)}, Future rows: {len(future)}")

Dataset end: 2011-12-09 12:50:00, Cutoff: 2011-09-10 12:50:00
Past rows: 620623, Future rows: 158872


3. Features sirf past se banao:

In [15]:
features = past.groupby('Customer ID').agg(
    Frequency=('Invoice', 'nunique'),
    Monetary=('LineTotal', 'sum'),
    UniqueProducts=('StockCode', 'nunique'),
    FirstPurchase=('InvoiceDate', 'min'),
    LastPurchase=('InvoiceDate', 'max'),
    Country=('Country', lambda s: s.mode()[0])
).reset_index()

features['Recency'] = (cutoff - features['LastPurchase']).dt.days
features['Tenure'] = (cutoff - features['FirstPurchase']).dt.days
features['AvgOrderValue'] = features['Monetary'] / features['Frequency']
features['Monetary_log'] = np.log1p(features['Monetary'])
features = features.drop(columns=['FirstPurchase', 'LastPurchase'])
print(features.head())

   Customer ID  Frequency  Monetary  UniqueProducts         Country  Recency  \
0        12346         12  77556.46              27  United Kingdom      235   
1        12347          6   3402.39             107         Iceland       39   
2        12348          4   1709.40              25         Finland      158   
3        12349          3   2671.14              90           Italy      317   
4        12350          1    334.40              17          Norway      219   

   Tenure  AvgOrderValue  Monetary_log  
0     635    6463.038333     11.258774  
1     313     567.065000      8.132527  
2     347     427.350000      7.444483  
3     498     890.380000      7.890635  
4     219     334.400000      5.815324  


Yahan Recency ka matlab hai “cutoff tak kitne din se inactive”. Yeh ab legit hai, kyunki label cutoff ke baad ka hai.

4. Label future se banao:

In [16]:
active_in_future = set(future['Customer ID'].unique())
features['Churned'] = (~features['Customer ID'].isin(active_in_future)).astype(int)

print(features['Churned'].value_counts())
print((features['Churned'].value_counts(normalize=True) * 100).round(2))

Churned
1    2991
0    2292
Name: count, dtype: int64
Churned
1    56.62
0    43.38
Name: proportion, dtype: float64


Yeh numbers Day 56B se alag aayenge, kyunki ab definition future window par hai. Imbalance ratio ek baar dobara nikaal lo (majority / minority) aur note karo, kyunki Day 64C mein wahi use hoga.

5. Region feature banao (Day 57 jaisa):

In [17]:
european = ['Germany', 'France', 'Spain', 'Netherlands', 'Belgium', 'Switzerland', 'Portugal',
            'Italy', 'Austria', 'Denmark', 'Sweden', 'Norway', 'Finland', 'EIRE', 'Ireland',
            'Poland', 'Channel Islands', 'Cyprus', 'Greece']

def region_bucket(c):
    if c == 'United Kingdom':
        return 'UK'
    return 'Rest of Europe' if c in european else 'Rest of World'

features['Region'] = features['Country'].apply(region_bucket)

6. Leakage sanity check (zaroor chalana):

In [18]:
print(features[['Recency', 'Frequency', 'Monetary_log', 'Tenure', 'Churned']].corr()['Churned'].round(3))

Recency         0.437
Frequency      -0.268
Monetary_log   -0.411
Tenure         -0.044
Churned         1.000
Name: Churned, dtype: float64


Agar Recency ka correlation moderate (jaise 0.3-0.5) hai, to sab theek hai. Agar ~0.9+ aaye, to kuch galat hai aur cutoff logic dobara check karo.

7. X aur y alag karo:

In [19]:
X = features[['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue', 'Tenure', 'UniqueProducts', 'Region']]
X = pd.get_dummies(X, columns=['Region'], drop_first=True)   # categorical ko numbers mein
y = features['Churned']

print(X.shape, y.shape)
print(X.columns.tolist())
print(X.isnull().sum().sum())   # 0 aana chahiye

(5283, 8) (5283,)
['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue', 'Tenure', 'UniqueProducts', 'Region_Rest of World', 'Region_UK']
0


Customer ID aur raw Country X mein nahi gaye: ID sirf ek label hai (usme koi pattern nahi), aur Region Country ka summarised version hai.

8. Save karo (Day 63 se yahi use hoga):

In [20]:
features.to_csv('../../data/processed/customer_ml_dataset.csv', index=False)
print(f"Saved: {features.shape}")

Saved: (5283, 11)


practice questions:

1. Cutoff ko 90 se 60 din karke dekho, churn % aur Recency ka correlation kaise badalte hain.

Agar aap cutoff ko 90 din se ghatakar 60 din kar dete hain (matlab prediction window ko chhota karke 60 days kar dete hain):

- Churn Percentage par asar: Churn rate kam (decrease) ho jayega. Kyunki aisi bahut saari customer populations hongi jo 60 din ke andar toh inactive rahi hongi (shayad 70 ya 80 din mein purchase karti hain), lekin 90 din ke window mein wapas aa jati hain. Jab window 60 din ki hogi, toh woh "churned" count hongi, lekin 90 din mein "active" count hoti thin. Chhota window matlab strict criteria, isliye kam log churned classify honge.


- Recency Correlation par asar: Recency (cutoff tak kitne din se inactive the) ka correlation Churned target ke sath stronger (aur high) ho jayega. Kyunki agar koi customer pichle 50 din se purchase nahi kiya, toh 60 din wale window mein uske churn hone ke chances bahut zyada high hain. Isliye recency ek powerful deterministic signal ban jata hai, jisse correlation thoda aur upar ja sakta hai.

2. Apne shabdon mein likho: purane setup (Day 35B) mein leakage kahan tha, aur naye setup mein kyun nahi hai?

- Purane Setup (Day 35B) mein Leakage:
Purane setup mein Churned label poore dataset ki timeline (aakhir tak) ko dekh kar banaya gaya tha aur usme Recency bhi usi poore timeframe ke aadhar par calculate hui thi. Iska matlab yeh tha ki model ko pehle hi pata tha ki kaun sa customer aage chalkar inactive hone wala hai. Jab model features (Recency, Monetary, etc.) aur label (Churned) ek hi same future/present mix window se dekhta hai, toh model "future information" ko cheat karke use kar leta hai—ise hi Data Leakage kehte hain. Model artificial tareeqe se 99% accuracy de deta hai kyunki usne actual learning nahi ki, balki future ka answer pehle hi padh liya hai.


- Naye Setup mein Leakage kyun nahi hai:
Naye setup mein humne strict Time-based Split (Cutoff Date) apply kiya hai:

    1. Features: Sirf aur sirf cutoff date se pehle (past) ke transactions se banaye gaye hain.
    
    2. Label (Churned): Sirf yeh check karta hai ki customer ne cutoff ke baad (future) ke 90 din mein order diya ya nahi.


    
Yahan model past ke behaviour ko dekhkar future predict karne ki koshish karta hai, theek waisa hi jaise real-world production ya live business mein hota hai. Kyunki future ka data features ke andar chupaya nahi gaya hai, isliye yahan koi data leakage nahi hai aur model ka score ya evaluation ek dam honest aur reliable hoga.